In [ ]:
# %pip install "lm_eval[ifeval]"

In [1]:
import torch, gc
from transformers import AutoModelForCausalLM, AutoTokenizer

CHECKPOINTS = {
    "base":     "allenai/OLMo-2-0425-1B",
    "sft":      "allenai/OLMo-2-0425-1B-SFT",
    "dpo":      "allenai/OLMo-2-0425-1B-DPO",
    "instruct": "allenai/OLMo-2-0425-1B-Instruct",
}
# for checking
def load(name):
    tok = AutoTokenizer.from_pretrained(name)
    model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.bfloat16).to("cuda")
    model.eval()
    return model, tok
# for cheking
def generate(model, tok, prompt, use_chat, max_new_tokens=256):
    if use_chat:
        text = tok.apply_chat_template([{"role": "user", "content": prompt}],
                                       tokenize=False, add_generation_prompt=True)
    else:
        text = prompt
    inputs = tok(text, return_tensors="pt", return_token_type_ids=False).to("cuda")
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)


C:\Users\ASUS\AppData\Roaming\Python\Python311\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.4.0) or chardet (6.0.0.post1)/charset_normalizer (3.4.2) doesn't match a supported version!
  warnings.warn(


In [2]:
import lm_eval, json

all_results, all_samples = {}, {}
for key, name in CHECKPOINTS.items():
    res = lm_eval.simple_evaluate(
        model="hf",                                         # run model through hugging face
        model_args=f"pretrained={name},dtype=bfloat16",
        tasks=["gsm8k", "ifeval"],                          # benchmarks chosen
        limit=200,                                          # same 200 examples
        apply_chat_template=(key != "base"),                # wrap each prompt in the model’s chat format (user/assistant turns) for SFT, DPO, and Instruct
        batch_size=8,
        device="cuda",
        log_samples=True,                                   # keep each example's prompt and output for extracting
    )
    all_results[key] = res["results"]
    all_samples[key] = res["samples"]
    print(key, res["results"])

    with open(f"results_{key}.json", "w") as f:
        json.dump({"results": res["results"], "samples": res["samples"]}, f, default=str)

    gc.collect()
    torch.cuda.empty_cache()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Generating train split:   0%|          | 0/7473 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1319 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/541 [00:00<?, ? examples/s]

Running generate_until requests: 100%|██████████| 400/400 [28:58<00:00,  4.35s/it]


base {'gsm8k': {'name': 'gsm8k', 'alias': 'gsm8k', 'sample_len': 200, 'exact_match,strict-match': np.float64(0.3), 'exact_match_stderr,strict-match': 0.03248501780682212, 'exact_match,flexible-extract': np.float64(0.3), 'exact_match_stderr,flexible-extract': 0.03248501780682212}, 'ifeval': {'name': 'ifeval', 'alias': 'ifeval', 'sample_len': 200, 'prompt_level_strict_acc,none': 0.075, 'prompt_level_strict_acc_stderr,none': 0.018671317319682135, 'inst_level_strict_acc,none': 0.18238993710691823, 'inst_level_strict_acc_stderr,none': 'N/A', 'prompt_level_loose_acc,none': 0.095, 'prompt_level_loose_acc_stderr,none': 0.02078545587374489, 'inst_level_loose_acc,none': 0.1949685534591195, 'inst_level_loose_acc_stderr,none': 'N/A'}}


... (2 outputs left out)

Running generate_until requests: 100%|██████████| 400/400 [22:47<00:00,  3.42s/it]


dpo {'gsm8k': {'name': 'gsm8k', 'alias': 'gsm8k', 'sample_len': 200, 'exact_match,strict-match': np.float64(0.47), 'exact_match_stderr,strict-match': 0.03538020341900045, 'exact_match,flexible-extract': np.float64(0.515), 'exact_match_stderr,flexible-extract': 0.03542810683297719}, 'ifeval': {'name': 'ifeval', 'alias': 'ifeval', 'sample_len': 200, 'prompt_level_strict_acc,none': 0.535, 'prompt_level_strict_acc_stderr,none': 0.035357115664894224, 'inst_level_strict_acc,none': 0.6792452830188679, 'inst_level_strict_acc_stderr,none': 'N/A', 'prompt_level_loose_acc,none': 0.59, 'prompt_level_loose_acc_stderr,none': 0.03486513859784927, 'inst_level_loose_acc,none': 0.720125786163522, 'inst_level_loose_acc_stderr,none': 'N/A'}}


Running generate_until requests: 100%|██████████| 400/400 [22:49<00:00,  3.42s/it]


instruct {'gsm8k': {'name': 'gsm8k', 'alias': 'gsm8k', 'sample_len': 200, 'exact_match,strict-match': np.float64(0.53), 'exact_match_stderr,strict-match': 0.03538020341900046, 'exact_match,flexible-extract': np.float64(0.62), 'exact_match_stderr,flexible-extract': 0.034408132730358205}, 'ifeval': {'name': 'ifeval', 'alias': 'ifeval', 'sample_len': 200, 'prompt_level_strict_acc,none': 0.585, 'prompt_level_strict_acc_stderr,none': 0.034928138718973496, 'inst_level_strict_acc,none': 0.7044025157232704, 'inst_level_strict_acc_stderr,none': 'N/A', 'prompt_level_loose_acc,none': 0.62, 'prompt_level_loose_acc_stderr,none': 0.0344081327303582, 'inst_level_loose_acc,none': 0.7358490566037735, 'inst_level_loose_acc_stderr,none': 'N/A'}}


In [5]:
for key in CHECKPOINTS:
    s = all_samples[key]["gsm8k"][67]
    print(f"--- {key} ---")
    print("Question:", s["doc"]["question"])
    print("Answer:", s["resps"][0][0][:500])
    print("Target:", s["target"])

--- base ---
Question: A treasure hunter found a buried treasure chest filled with gems. There were 175 diamonds, 35 fewer rubies than diamonds, and twice the number of emeralds than the rubies.   How many of the gems were there in the chest?
Answer:  There were 175 - 35 = <<175-35=140>>140 rubies.
There were 2 * 140 = <<2*140=280>>280 emeralds.
Therefore, there were 140 + 280 + 175 = <<140+280+175=555>>555 gems in the chest.
#### 555
Target: There were 175 - 35 = <<175-35=140>>140 rubies in the treasure chest.
There were 140 x 2 = <<140*2=280>>280 emeralds in the treasure chest.
So, the total number of gems in the treasure chest is 175 + 140 + 280 = <<175+140+280=595>>595.
#### 595
--- sft ---
Question: A treasure hunter found a buried treasure chest filled with gems. There were 175 diamonds, 35 fewer rubies than diamonds, and twice the number of emeralds than the rubies.   How many of the gems were there in the chest?
Answer: There were 175 - 35 = <<175-35=140>>140 rubies.
There were

In [7]:
for key in CHECKPOINTS:
    s = all_samples[key]["ifeval"][67]
    print(f"--- {key} ---")
    print("Prompt:", s["doc"]["prompt"])
    print("Instructions:", s["doc"]["instruction_id_list"])
    print("Strictly Followed:", s["inst_level_strict_acc"],
          "| all followed:", s["prompt_level_strict_acc"])
    print("Answer:", s["resps"][0][0][:500])

--- base ---
Prompt: Write a brief biography of a person named "Brilliant Le Hou". Do not use commas in your reply. Highlight at least 3 sections with markdown, i.e. *highlighted section*. The biography should start with the name of the person.
Instructions: ['punctuation:no_comma', 'detectable_format:number_highlighted_sections', 'keywords:existence']
Strictly Followed: [False, False, False] | all followed: False
Answer:  You can then write about their achievements, education, and other details. You can also include some facts and figures to make it more interesting. You can also include some pictures or videos that will help people understand the person better. You can also include some links to websites or articles that will provide more information about the person. You can also include some questions that will help people learn more about the person. You can also include some answers to those questions. You 
--- sft ---
Prompt: Write a brief biography of a person named "Brilliant 

In [11]:
import json
from collections import Counter

def is_loop(text, n=4, threshold=0.3):
    words = text.split()
    grams = [tuple(words[i:i + n]) for i in range(len(words) - n + 1)]
    if len(grams) < 20:
        return False
    return Counter(grams).most_common(1)[0][1] / len(grams) > threshold

rows = []
for key in CHECKPOINTS:
    with open(f"results_{key}.json") as f:
        d = json.load(f)
    r = d["results"]

    gsm = [s["resps"][0][0] for s in d["samples"]["gsm8k"] if s["filter"] == "strict-match"]
    ife = [s["resps"][0][0] for s in d["samples"]["ifeval"]]

    rows.append({
        "checkpoint": key,
        "gsm8k": 100 * float(r["gsm8k"]["exact_match,strict-match"]),
        "ife_all": 100 * float(r["ifeval"]["prompt_level_strict_acc,none"]),
        "ife_indiv": 100 * float(r["ifeval"]["inst_level_strict_acc,none"]),
        "gsm_words": sum(len(t.split()) for t in gsm) / len(gsm),
        "ife_words": sum(len(t.split()) for t in ife) / len(ife),
    })

print(f"{'checkpoint':<10} {'GSM8K':>6} {'IFE prompt':>11} {'IFE inst':>9} "
      f"{'GSM words':>10} {'IFE words':>10}")
for x in rows:
    print(f"{x['checkpoint']:<10} {x['gsm8k']:6.1f} {x['ife_all']:11.1f} {x['ife_indiv']:9.1f} "
          f"{x['gsm_words']:10.0f} {x['ife_words']:10.0f}")

checkpoint  GSM8K  IFE prompt  IFE inst  GSM words  IFE words
base         30.0         7.5      18.2         50        610
sft          34.0        43.0      55.3         64        272
dpo          47.0        53.5      67.9         77        251
instruct     53.0        58.5      70.4         98        239
